# training ANPEs on adroit

In [1]:
import os, sys 
import numpy as np

import torch
from torch import nn 
from torch.utils.tensorboard.writer import SummaryWriter

from sbi import utils as Ut
from sbi import inference as Inference

from sedflow import data as D
from sedflow import util as U

In [2]:
cuda = torch.cuda.is_available()
device = ("cuda:0" if cuda else "cpu")

seed = 12387
torch.manual_seed(seed)
if cuda:
    torch.cuda.manual_seed(seed)

In [3]:
bands = 'ugrizJ'
freez = False

In [4]:
x_train, y_train = D.load_modela('train', bands=bands, infer_redshift=freez)
print('Ntrain + Nvalid = %i' % (x_train.shape[0]))

Ntrain + Nvalid = 1000000


In [5]:
prior_low   = [7, 0., 0., 0., 0., 1e-2, np.log10(4.5e-5), np.log10(4.5e-5), 0, 0., -2.]
prior_high  = [12.5, 1., 1., 1., 1., 13.27, np.log10(1.5e-2), np.log10(1.5e-2), 3., 3., 1.]

lower_bounds = torch.tensor(prior_low).to(device)
upper_bounds = torch.tensor(prior_high).to(device)

prior = Ut.BoxUniform(low=lower_bounds, high=upper_bounds, device=device)

/home/chhahn/.conda/envs/torch-env/lib/python3.7/site-packages/sbi/utils/torchutils.py:28: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  "GPU was selected as a device for training the neural network. "


In [6]:
neural_posterior = Ut.posterior_nn('maf', 
        hidden_features=500, 
        num_transforms=10,   
        use_batch_norm=True)

In [7]:
anpe = Inference.SNPE(prior=prior,
        density_estimator=neural_posterior,
        device=device)

In [8]:
anpe.append_simulations(
    torch.as_tensor(x_train.astype(np.float32)).to(device),
    torch.as_tensor(y_train.astype(np.float32)).to(device))

In [ ]:
p_x_y_est = anpe.train()

 Training neural network. Epochs trained: 4